Benchmarking - Concepts
=======================

**Topic**: How to compare path planning algorithms systematically

Version | Author
------------ | -------------
0.2 | Björn Hein

This worksheet explains **how planners can be compared**: what a benchmark is, how benchmarks are collected in a test suite, which data is meaningful, how this data is recorded (profiling, decorators, `IPPerfMonitor`) and how it is evaluated with several runs per planner.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-4-0-Test-Environments (collision checker, benchmark scenes, clearance) and IP-5-1-PRM-Basics-class (the planner `BasicPRM`, which is used as example)

**Dependencies**:

* IPBenchmark.py (class `Benchmark`), IPTestSuite.py (the benchmarks), IPPerfMonitor.py (recording of calls), IPBasicPRM.py and IPVISBasicPRM.py (example planner and its visualization), IPVISStyle.py (colors)

Important links are:

* General Info: https://www.python.org
* NetworkX: https://networkx.org
* pandas: https://pandas.pydata.org/docs/
* Matplotlib: https://matplotlib.org/stable/
* Profiling (profile, cProfile, pstats): https://docs.python.org/3/library/profile.html
* timeit: https://docs.python.org/3/library/timeit.html

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Goal

* Understanding why planners have to be compared on well defined problems (benchmarks) and with several runs
* Understanding how benchmarks are defined (`Benchmark`) and collected in a test suite (`IPTestSuite.py`)
* Understanding which data is meaningful: success rate, planning time, number of collision checks, path length, size of the roadmap
* Understanding how to find out where the time is spent (profiling, micro-benchmarks with `timeit`)
* Understanding decorators and how `IPPerfMonitor` uses them to record the calls of planners and collision checkers
* Understanding how to evaluate the recorded data with pandas

# Idea

Which planner is "better"? A single picture of a single run cannot answer this question:

* The answer depends on the **problem**: a planner may be fast in an almost empty scene and fail in a narrow passage.
* The answer depends on what is **important**: finding a path at all, finding it fast, or finding a short path.
* The planners of this course are **randomized**: the same planner with the same parameters finds a path in one run and fails in the next one.

Therefore planners are compared systematically:

1. **Benchmarks**: well defined planning problems (scene, start, goal), which are the same for all planners. A collection of benchmarks is a **test suite**.
2. **Planner configurations**: a planner class plus its parameters, e.g. `BasicPRM` with radius 3 and 150 nodes. The same planner with other parameters is another configuration.
3. **Runs**: every configuration is run on every benchmark, a randomized planner several times (with different, but fixed seeds of the random generator, so the results are reproducible).
4. **Measurements**: during every run data is recorded.
5. **Evaluation**: the data of all runs is aggregated (success rate, mean, spread) and visualized.

Typical measures:

Measure | Meaning | Remark
------ | ------- | ------
success rate | fraction of the runs, in which a path was found | the most important measure for randomized planners
planning time | time of `planPath` | depends on the computer, the implementation and the load of the machine
number of collision checks | calls of `pointInCollision` and `lineInCollision` | independent of the computer; usually the dominating cost (see IP-4-0)
path length | length of the solution path | quality of the solution; only defined for successful runs
size of the roadmap / tree | number of nodes and edges | memory consumption, effort of the graph search

# Benchmarks

## Defining a benchmark

A benchmark bundles everything which defines a planning problem (see IP-4-0). The class `Benchmark` from IPBenchmark.py is only a container:

In [ ]:
import inspect
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import LineString

from IPBenchmark import Benchmark
from IPEnvironment import CollisionChecker
from IPVISStyle import START_COLOR, GOAL_COLOR

print(inspect.getsource(Benchmark))

* `startList` and `goalList` are **lists** of configurations: a planner may use any of them. (The planners of this course use the first valid start and the first valid goal.)
* `level` is the assumed difficulty from 1 ("easy") to 4 ("insane"). In IPTestSuite.py it is derived from the clearance of the benchmark (see IP-4-0).
* A benchmark knows nothing about the planners. Every planner gets the same collision checker, start and goal: all planners solve exactly the same problem.

Example: the benchmark "Trap" (see IP-4-0). The direct line from start to goal leads into the "U".

In [ ]:
limits = [[0.0, 22.0], [0.0, 22.0]]

trapField = dict()
trapField["obs1"] = LineString([(6, 18), (6, 8), (16, 8), (16, 18)]).buffer(1.0)
description = "Following the direct connection from goal to start would lead the algorithm into a trap."
trapBenchmark = Benchmark("Trap", CollisionChecker(trapField, limits), [[10, 15]], [[10, 1]], description, 1)

Two small helper functions for the drawings of this notebook (obstacles in gray, start in green, goal in red, as in all notebooks):

In [ ]:
def setupAxes(ax, benchmark, title):
    limits = benchmark.collisionChecker.getEnvironmentLimits()
    ax.set_xlim(limits[0])
    ax.set_ylim(limits[1])
    ax.set_aspect("equal")
    ax.set_title(title)


def drawBenchmark(benchmark, ax=None, figsize=(4.5, 4.5)):
    # Obstacles, start and goal configurations of a benchmark
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    benchmark.collisionChecker.drawObstacles(ax)
    ax.scatter(*zip(*benchmark.startList), color=START_COLOR, s=80, zorder=5)
    ax.scatter(*zip(*benchmark.goalList), color=GOAL_COLOR, s=80, zorder=5)
    setupAxes(ax, benchmark, f"{benchmark.name} (level {benchmark.level})")
    return ax


ax = drawBenchmark(trapBenchmark)

## Collecting benchmarks in a test suite

Benchmarks are collected in a list. To use the same benchmarks in all notebooks, this list is defined in a module: IPTestSuite.py contains the list `benchList` with the benchmark "Trap" from above and further benchmarks (their levels were determined in IP-4-0). Open the file to see the definitions.

Advantages of a test suite in a module:

* All planners and all notebooks use **the same** problems: the results are comparable.
* A new benchmark is added at one place and is automatically used by all comparisons (e.g. in IP-X-1).
* The benchmarks are validated once (see `validateBenchmark` in IP-4-0): valid start and goal, solvable.

In [ ]:
import importlib
import IPTestSuite as ts
importlib.reload(ts)   # reloads the module, if IPTestSuite.py was changed while the kernel is running

for benchmark in ts.benchList:
    print(f"{benchmark.name:15s} level {benchmark.level}: {benchmark.description}")

In [ ]:
fig, axes = plt.subplots(1, len(ts.benchList), figsize=(4.2 * len(ts.benchList), 4.2))
for ax, benchmark in zip(axes, ts.benchList):
    drawBenchmark(benchmark, ax=ax)

## Running a planner on all benchmarks

The Basic PRM (IP-5-1) with one configuration on all benchmarks. As for all planners of the course the parameters are passed as dict. An error in one benchmark (e.g. an invalid start) must not stop the loop over all benchmarks, therefore every run is protected by `try`/`except`.

In [ ]:
from IPBasicPRM import BasicPRM
from IPVISBasicPRM import basicPRMVisualize

basicConfig = dict()
basicConfig["radius"] = 3
basicConfig["numNodes"] = 150

fig, axes = plt.subplots(1, len(ts.benchList), figsize=(4.2 * len(ts.benchList), 4.2))
for ax, benchmark in zip(axes, ts.benchList):
    try:
        planner = BasicPRM(benchmark.collisionChecker)
        solution = planner.planPath(benchmark.startList, benchmark.goalList, basicConfig)
        result = "path found" if solution else "no path found"
        basicPRMVisualize(planner, solution, ax=ax, nodeSize=20)
        setupAxes(ax, benchmark, f"{benchmark.name} (level {benchmark.level})\n{result}")
    except Exception as e:
        print("ERROR:", benchmark.name, e)

Execute the cell several times: the results change from run to run. A single run tells little about a randomized planner. Therefore the data of many runs is recorded and evaluated (see "Statistics over several runs" below). But first: how is the data recorded?

# Collecting data

## Measuring the planning time

The simplest measurement is the duration of a call. `time.perf_counter()` is the clock with the highest resolution for measuring durations (`time.time()` returns the time of day; it may jump, e.g. when the system clock is synchronized).

Five runs of the Basic PRM on "Trap" with a larger roadmap:

In [ ]:
trapConfig = dict()
trapConfig["radius"] = 3
trapConfig["numNodes"] = 300

times = []
for run in range(5):
    planner = BasicPRM(trapBenchmark.collisionChecker)
    startTime = time.perf_counter()
    solution = planner.planPath(trapBenchmark.startList, trapBenchmark.goalList, trapConfig)
    times.append(time.perf_counter() - startTime)
    print(f"run {run}: {times[-1]:.3f} s, path found: {len(solution) > 0}")

The time varies from run to run, even for the same problem: different random samples, warm-up effects (the first run is often slower), other processes on the computer, the garbage collector of Python, ... Times are only comparable when measured on the same computer, and several runs are needed.

And the time alone does not tell **where** it is spent.

## Profiling with profile/cProfile and pstats

A **profiler** records every function call during the execution of a program: how often a function is called and how much time is spent in it. Python includes two profilers with the same interface, and the module `pstats` to evaluate their results (https://docs.python.org/3/library/profile.html):

* `cProfile` is implemented in C and slows down the program only moderately. It is the first choice - **in a Python script**, e.g. `python -m cProfile -s cumulative script.py`.
* `profile` is implemented in Python and slows down the program much more.

In a Jupyter notebook `profile` is used here: since Python 3.12, `cProfile` records the function calls of **all** threads of the process. The Jupyter kernel runs background threads (e.g. for sending the outputs of the cells), whose calls get mixed into the profile and randomly corrupt the cumulative times.

The method `runcall` of the profiler executes a function with the given arguments under the control of the profiler and returns its result. The seed of the random generator is fixed, so the same roadmap is generated every time the cell is executed.

In [ ]:
import profile
import pstats
from pstats import SortKey

random.seed(1)
planner = BasicPRM(trapBenchmark.collisionChecker)
profiler = profile.Profile()
startTime = time.perf_counter()
solution = profiler.runcall(planner.planPath, trapBenchmark.startList, trapBenchmark.goalList, trapConfig)
profiledTime = time.perf_counter() - startTime

stats = pstats.Stats(profiler).strip_dirs()   # strip_dirs: file names without directories
stats.sort_stats(SortKey.CUMULATIVE).print_stats(12)

The columns:

* `ncalls`: number of calls. Two numbers (e.g. `15883/1`) appear for recursive functions: all calls / calls, which were not made while the function was already active.
* `tottime`: time spent in the function itself, **without** the functions called by it
* `cumtime`: time spent in the function **including** all functions called by it
* `percall`: `tottime` resp. `cumtime` divided by the number of calls

Sorted by `cumtime` the list shows the call hierarchy from top to bottom: `planPath` needs all the time, almost all of it in `_learnRoadmapNearestNeighbour`. The first line is the function passed to `runcall`: `planner.planPath` is not a plain method, but a `functools.partial` object of `IPPerfMonitor`. `IPPerfMonitor.py(__call__)` appears on top and looks recursive, because the course modules call some functions through it, and these functions call further ones through it (both see "Decorators" below).

A restriction (a regular expression on file names and function names) shows only the functions of the course modules:

In [ ]:
stats.sort_stats(SortKey.CUMULATIVE).print_stats(r"IP\w*\.py")

Surprisingly, the collision checks are **not** the most expensive part: `_nearestNeighbours` needs more time than `lineInCollision`. It searches the neighbours by brute force: the distance of the new node to every node of the roadmap is computed with `euclidean` from SciPy. Sorted by `tottime`, the functions which consume the time themselves are shown:

In [ ]:
stats.sort_stats(SortKey.TIME).print_stats(10)

Most functions of this list are called by `euclidean` (`minkowski`, `norm`, `asarray`, ...): on every call SciPy checks and converts its arguments to NumPy arrays, which costs much more than computing the distance of two 2D points. The functions from `decorators.py` and `point.py` belong to Shapely, i.e. to the collision checks.

The profiler itself slows down the program:

In [ ]:
print(f"with profiler:    {profiledTime:.3f} s")
print(f"without profiler: {min(times):.3f} s (fastest run of the measurement above)")

Remarks:

* Because of this overhead, the absolute times of a profile are not meaningful. The overhead is added to every function call: functions, which consist of many small calls (like `euclidean`), appear more expensive than they are. The proportions are therefore only a hint, which has to be checked (see below).
* Profiling is a tool for the **developer** to find the bottleneck ("hot spot") of an implementation. For comparing planners it is too detailed and too slow.

## Micro-benchmarks with timeit

If a single function is suspected, alternatives can be compared with a **micro-benchmark**: the module `timeit` executes a small piece of code many times and measures the total time. `autorange` determines how many executions take at least 0.2 s. The measurement is repeated and the minimum is taken: it is the measurement least disturbed by other processes. (In a notebook, the magic command `%timeit euclidean(p, q)` does the same in one line.)

Three ways to compute the distance of two 2D points:

In [ ]:
import timeit
from scipy.spatial.distance import euclidean

p, q = [3.2, 4.5], [10.1, 7.7]
candidates = {"scipy.spatial.distance.euclidean": "euclidean(p, q)",
              "numpy.linalg.norm": "np.linalg.norm(np.subtract(p, q))",
              "math.dist": "math.dist(p, q)"}

for name, statement in candidates.items():
    timer = timeit.Timer(statement, globals=globals())
    number, _ = timer.autorange()
    best = min(timer.repeat(repeat=5, number=number)) / number
    print(f"{name:34s} {best * 1e6:6.2f} µs per call (result: {eval(statement):.4f})")

All three compute the same distance, but `math.dist` is more than an order of magnitude faster. SciPy and NumPy are made for large arrays; for two 2D points the overhead of each call dominates. Following the profile, `euclidean` in `_nearestNeighbours` should be replaced (see exercises), or a better algorithm should be used: a k-d tree (as in IPRRT.py) finds the neighbours without computing all distances.

**Important**: a micro-benchmark measures only the function itself. Whether a change pays off for the whole planner has to be measured with the whole planner again - and the planner must still compute the same result.

## Decorators

For the comparison of planners not every function call has to be recorded, but only the important ones: the collision checks, `planPath` and some characteristic functions of each planner. The course modules do this with a **decorator**.

### Functions are objects

In Python a function is an object: it can be assigned to a variable, passed to a function and returned by a function.

In [ ]:
def just_some_function():
    print("Super Duper!")

just_some_function()

another_name = just_some_function   # no call (no brackets): a second name for the same function object
another_name()

### A decorator wraps a function

A decorator is a function, which gets a function and returns a new function: the **wrapper**. The wrapper calls the original function and can do something before and after the call:

In [ ]:
def my_decorator(some_function):
    print("Info - I was called")
    counter = 0

    def wrapper():
        nonlocal counter
        counter += 1
        print("Something is happening before some_function() is called.", counter)
        some_function()
        print("Something is happening after some_function() is called.", counter)

    return wrapper

* `wrapper` is defined inside `my_decorator` and uses its variables `some_function` and `counter`: it is a **closure**. These variables live on as long as the wrapper exists, even after `my_decorator` has returned.
* `nonlocal counter` is needed, because `counter` is assigned inside `wrapper`. Without it, Python would treat `counter` as a new local variable of `wrapper` (and raise an `UnboundLocalError` at `counter += 1`).
* Every call of `my_decorator` creates a new wrapper with its own counter.

The decorator is applied by replacing the function with the wrapper:

In [ ]:
just_some_function = my_decorator(just_some_function)

In [ ]:
for i in range(3):
    just_some_function()

### Syntactic sugar: @

The `@` notation does exactly the same: `@my_decorator` above a definition means `just_some_function = my_decorator(just_some_function)` directly after the definition.

In [ ]:
@my_decorator
def just_some_function():
    print("Super Duper!")

In [ ]:
for i in range(3):
    just_some_function()

**Please take note:** "Info - I was called" is printed when the function is **defined**: in the notebook when the cell is executed, in a module when the module is imported. The decorator itself is executed only once per decorated function, the wrapper on every call. The counter starts again at 1, because the new definition got a new wrapper.

### A decorator for measurements

A decorator for arbitrary functions has to pass on all arguments and the return value: `*args` collects all positional arguments in a tuple, `**kwargs` all keyword arguments in a dict. `functools.wraps` copies the name and the docstring of the original function to the wrapper.

The following decorator records arguments, return value and duration of every call in a list:

In [ ]:
import functools

def record(function):
    calls = []

    @functools.wraps(function)
    def wrapper(*args, **kwargs):
        startTime = time.perf_counter()
        result = function(*args, **kwargs)
        calls.append({"args": args, "kwargs": kwargs, "retVal": result, "time": time.perf_counter() - startTime})
        return result

    wrapper.calls = calls   # functions are objects: the list is attached as attribute
    return wrapper


@record
def slowSquare(x, pause=0.01):
    "Square of x (with a short break)"
    time.sleep(pause)
    return x * x

for x in range(4):
    slowSquare(x)
slowSquare(5, pause=0.05)

print(slowSquare.__name__, "-", slowSquare.__doc__)
pd.DataFrame(slowSquare.calls)

This is already the principle of `IPPerfMonitor`.

### The decorator class IPPerfMonitor

`IPPerfMonitor` does the same, but it is a **class**: decorating a function creates an instance of the class, which replaces the function. Calling the "function" then calls the method `__call__` of this instance.

In [ ]:
from IPPerfMonitor import IPPerfMonitor

print(inspect.getsource(IPPerfMonitor))

* `__init__` is executed once, when the function is decorated (when the module is imported). It stores the original function and registers the new instance in the class attribute `__instances`: a dict shared by **all** decorated functions.
* `__call__` is executed on every call: it calls the original function, measures the time and appends arguments, return value and duration to the list `self.data`.
* `__get__` is needed for **methods** (see below).
* `dataFrame()` collects the data of all registered functions in one pandas `DataFrame`; the column `name` contains the name of the function. `clearData()` deletes the data of all functions.

### Why `__get__`?

When a method is accessed via an instance, e.g. `checker.pointInCollision`, Python creates a *bound method*: the instance is passed automatically as first argument `self`. Normal functions provide this with their method `__get__` (descriptor protocol). An instance of a decorator class is not a function: without `__get__` the argument `self` is missing. A small experiment:

In [ ]:
class CallCounter(object):
    "Decorator class without __get__"

    def __init__(self, f):
        self.f = f
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.f(*args, **kwargs)


@CallCounter
def double(x):
    return 2 * x

print("function:", double(21), "- number of calls:", double.count)


class Example(object):

    @CallCounter
    def double(self, x):
        return 2 * x

try:
    Example().double(21)
except TypeError as e:
    print("method: TypeError:", e)

`Example().double(21)` calls `__call__(21)`, so `21` becomes `self` and `x` is missing. `__get__` gets the instance and returns `__call__` with the instance already filled in as first argument (`functools.partial`), exactly as in `IPPerfMonitor`. (That is why the profile above showed a `functools.partial` object for `planner.planPath`.)

In [ ]:
class MethodCallCounter(CallCounter):
    "Decorator class with __get__: works for methods, too"

    def __get__(self, instance, owner):
        return functools.partial(self.__call__, instance)


class Example(object):

    @MethodCallCounter
    def double(self, x):
        return 2 * x

example = Example()
print("method:", example.double(21), example.double(4), "- number of calls:", Example.__dict__["double"].count)

### Which functions are recorded?

The registry `__instances` contains all functions decorated with `@IPPerfMonitor` in the modules imported so far. (Attributes beginning with two underscores are renamed by Python to `_IPPerfMonitor__instances` to protect them from accidental use outside the class, *name mangling*. Here they are read only for the overview.)

In [ ]:
registered = {function.__qualname__: function for function in IPPerfMonitor._IPPerfMonitor__instances}
for qualname, function in registered.items():
    print(f"{function.__module__:15s} {qualname}")

The decorator is simply written above the definition, e.g. in IPBasicPRM.py:

In [ ]:
print(inspect.getsource(registered["BasicPRM._nearestNeighbours"]))

### Recording a planner run

`IPPerfMonitor` collects the data of all calls globally (also of previous runs!). Therefore the data has to be deleted with `clearData()` before every run. Then the planner is executed as usual:

In [ ]:
random.seed(1)
IPPerfMonitor.clearData()

planner = BasicPRM(trapBenchmark.collisionChecker)
solution = planner.planPath(trapBenchmark.startList, trapBenchmark.goalList, trapConfig)

fig, ax = plt.subplots(figsize=(5, 5))
basicPRMVisualize(planner, solution, ax=ax, nodeSize=30)
setupAxes(ax, trapBenchmark, f"{trapBenchmark.name}: path found: {len(solution) > 0}")

One row per recorded call. The column `args` contains all arguments; for methods the first one is the instance (`self`).

In [ ]:
df = IPPerfMonitor.dataFrame()
df.head()

Grouped by the name of the function: number of calls, total and mean time.

In [ ]:
df.groupby("name")["time"].agg(["count", "sum", "mean"])

Remarks:

* The **numbers of calls** are exact and do not depend on the computer. With the same seed they are the same in every execution of the cells above.
* The times are **inclusive**: the time of `lineInCollision` contains the time of the `pointInCollision` calls inside it, `planPath` contains everything. Therefore the times of different functions must not be added: their sum is larger than the planning time.
* The measurement itself costs time (the call of the decorator, storing the data): the times of functions, which are called very often and are fast, like `pointInCollision`, are too large.
* Same result as with the profiler: `_nearestNeighbours` needs more time than `lineInCollision`.

Filters on the columns answer more detailed questions, e.g. how many of the checks found a collision:

In [ ]:
for name in ["pointInCollision", "lineInCollision"]:
    checks = df[df.name == name]
    print(f"{name:16s}: {len(checks):5d} calls, {(checks.retVal == True).sum():4d} of them in collision")

# Statistics over several runs

Now everything is available for a systematic comparison: every configuration on every benchmark, several runs with fixed seeds. `runBenchmark` executes one run and returns the measured values as dict: one row of the result table.

In [ ]:
def pathLength(planner, solution):
    "Length of the solution path: sum of the lengths of its edges"
    nodes = planner.graph.nodes
    return sum(math.dist(nodes[a]["pos"], nodes[b]["pos"]) for a, b in zip(solution[:-1], solution[1:]))


def runBenchmark(plannerClass, config, benchmark, seed):
    "One run of a planner configuration on a benchmark. Returns the measured values as dict."
    random.seed(seed)
    IPPerfMonitor.clearData()
    planner = plannerClass(benchmark.collisionChecker)
    try:
        solution = planner.planPath(benchmark.startList, benchmark.goalList, config)
    except Exception as e:
        print("ERROR:", benchmark.name, e)
        solution = []
    calls = IPPerfMonitor.dataFrame()
    pointChecks = calls[calls.name == "pointInCollision"]
    return {"success": len(solution) > 0,
            "planning time": calls[calls.name == "planPath"]["time"].sum(),
            "point checks": len(pointChecks),
            "line checks": (calls.name == "lineInCollision").sum(),
            "µs per point check": pointChecks["time"].mean() * 1e6,
            "nodes": planner.graph.number_of_nodes(),
            "path length": pathLength(planner, solution) if solution else float("nan")}

Two configurations of the Basic PRM, 20 runs each on every benchmark (the cell takes about a minute):

In [ ]:
configurations = dict()
configurations["BasicPRM r=3 n=150"] = (BasicPRM, {"radius": 3, "numNodes": 150})
configurations["BasicPRM r=5 n=300"] = (BasicPRM, {"radius": 5, "numNodes": 300})
RUNS = 20

rows = []
for configName, (plannerClass, config) in configurations.items():
    for benchmark in ts.benchList:
        for seed in range(RUNS):
            row = {"configuration": configName, "benchmark": benchmark.name, "seed": seed}
            row.update(runBenchmark(plannerClass, config, benchmark, seed))
            rows.append(row)

results = pd.DataFrame(rows)
results.head()

Aggregated per benchmark and configuration:

In [ ]:
summary = results.groupby(["benchmark", "configuration"], sort=False).agg(
    success_rate=("success", "mean"),
    time_mean=("planning time", "mean"), time_std=("planning time", "std"),
    point_checks=("point checks", "mean"), line_checks=("line checks", "mean"),
    us_per_point_check=("µs per point check", "mean"),
    path_length=("path length", "mean"))
summary.round(3)

Remarks on the aggregation:

* The mean of a column with `True`/`False` is the fraction of `True`: the success rate.
* The mean path length is computed only from the successful runs (pandas ignores `NaN`). A configuration, which solves only the easy runs, may therefore get a "better" mean path length. Path lengths should only be compared together with the success rate.
* With a finite number of runs the success rate itself is uncertain: its standard error is $\sqrt{p(1-p)/n}$, for $p = 0.3$ and $n = 20$ runs about $0.1$. Small differences are not significant.

The same data as figure: success rate, and the distribution of planning time and point checks over the runs (box: middle half of the runs, line: median, whiskers: range without outliers).

In [ ]:
from IPVISStyle import PATH_COLOR, COMPONENT_COLOR, EDGE_COLOR

configColors = dict(zip(configurations, [PATH_COLOR, COMPONENT_COLOR]))   # the color belongs to the configuration
benchmarkNames = [benchmark.name for benchmark in ts.benchList]
x = np.arange(len(benchmarkNames))
width = 0.8 / len(configColors)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for i, (configName, color) in enumerate(configColors.items()):
    offset = (i - (len(configColors) - 1) / 2) * width
    data = results[results.configuration == configName]
    successRate = data.groupby("benchmark")["success"].mean().reindex(benchmarkNames)
    axes[0].bar(x + offset, successRate, width * 0.9, color=color, label=configName)
    for ax, column in zip(axes[1:], ["planning time", "point checks"]):
        values = [data[data.benchmark == name][column] for name in benchmarkNames]
        ax.boxplot(values, positions=x + offset, widths=width * 0.8, patch_artist=True,
                   boxprops=dict(facecolor=color, edgecolor=color), medianprops=dict(color=EDGE_COLOR, linewidth=2),
                   whiskerprops=dict(color=color), capprops=dict(color=color),
                   flierprops=dict(markeredgecolor=color, markersize=4))

for ax, label in zip(axes, ["success rate", "planning time [s]", "point checks per run"]):
    ax.set_xticks(x, benchmarkNames)
    ax.set_ylabel(label)
    ax.grid(axis="y", linewidth=0.3)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)
axes[0].set_ylim(0, 1.05)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=len(configColors), frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.93))

Observations:

* The success rate depends strongly on the benchmark. With radius 3 and 150 nodes the PRM solves "Trap" and "MyField" in three quarters of the runs, the narrow passages only rarely ("Fat bottleneck" in 2 of 20 runs). This agrees with the levels determined in IP-4-0. With radius 5 and 300 nodes almost every run succeeds - at about three times the planning time.
* For one configuration the number of point checks is almost the same on all benchmarks: the Basic PRM always builds the complete roadmap, the number of checks is determined by the number of nodes and the radius.
* The planning time, however, differs: in "MyField" a point check takes about six times as long as in "Trap". `pointInCollision` tests the point against every obstacle of the scene: "MyField" consists of eight obstacles, "Trap" of one. In "Trap" the point checks need less than half of the planning time (the rest is mainly the neighbour search), in "MyField" more than 80 %.
* Therefore: the planning time measures the planner **and** the cost of the collision checks (and the computer), the number of collision checks measures only the planner. A comparison needs both.
* The planning time and the number of checks scatter only little from run to run, the success varies. With radius 5 and 300 nodes the paths are a little shorter on all benchmarks (but see the remark above on the mean path length).

# Outlook: automated planner tests

This notebook used one planner. IP-X-1-Automated_PlanerTest applies the same concept to all planners of the course: the dict `plannerFactory` contains for every configuration the planner class, its parameters and its visualization function; all configurations are run on all benchmarks of IPTestSuite.py, and the results (planner, solution and the data of `IPPerfMonitor`) are collected for the evaluation.

# Exercises

1. Add your own benchmark (e.g. from the exercises of IP-4-0) to a copy of the test suite (`myBenchList = list(ts.benchList) + [...]`) and include it in the statistics.
2. Which parameter of the Basic PRM increases the success rate in "Bottleneck" more efficiently: the radius or the number of nodes? Add configurations and answer the question with the statistics (success rate **and** number of collision checks).
3. Replace `euclidean` in `BasicPRM._nearestNeighbours` (IPBasicPRM.py) by `math.dist`, restart the kernel and measure the planning time again. Check with fixed seeds that success, numbers of collision checks and path lengths stay the same: the benchmarks serve as regression test.
4. Add the planners `RRTSimple` and `RRT` from IPRRT.py (parameters see IP-9-0) to `configurations`. Which measures can be compared between PRM and RRT, which not (e.g. the number of nodes)?
5. Compare planners with a fixed **budget** instead of fixed parameters, e.g. stop after a fixed number of collision checks. Why is this a fairer comparison of planners with different parameters?
6. A PRM can answer many queries with one roadmap (multi-query), an RRT answers one query (single-query). How should a benchmark look like, which takes this into account?